# Phase 1 — Data prep (GenVidBench → frame shards → CLIP features)

Resumable: if Colab disconnects, re-run all cells; finished archives and shards are skipped.
Raw video only ever lives on Colab's local disk; Drive gets compact shards.

**Before you start:** copy your local `data/manifests/train.csv`, `val.csv`, `test.csv` to
`MyDrive/prism_video/manifests/` (same splits as EXP-004). Runtime → GPU (T4) only matters for step 5.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/prism_video'
import os
for d in ['frames', 'features', 'manifests', 'checkpoints', 'reports']:
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print(os.listdir(f'{DRIVE}/manifests'))

In [ ]:
!rm -rf /content/repo
!git clone -q -b claude/quirky-goodall-3ny0im https://github.com/rockrk2405/samsung_prism_project_1 /content/repo
%cd /content/repo/Samsung_PRISM_Video_Detection
!apt-get -qq install -y unar > /dev/null
!pip -q install -U huggingface_hub transformers opencv-python-headless

## 1. List archives in GenVidBench

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
REPO = 'jian-0/GenVidBench'
files = HfApi().list_repo_files(REPO, repo_type='dataset')
for f in files: print(f)

## 2. Pair1: download → extract → shard → delete, one archive at a time
Check the printed list above; adjust `PAIR1` if archive names differ (e.g. split `.part` files).

In [ ]:
import glob, shutil, subprocess, json
MANIFESTS = [f'{DRIVE}/manifests/{s}.csv' for s in ['train', 'val', 'test']]
DONE_LOG = f'{DRIVE}/frames/_archives_done.json'
done = set(json.load(open(DONE_LOG))) if os.path.exists(DONE_LOG) else set()

def process(archive_rel, folder, manifests):
    if archive_rel in done:
        print('skip (done):', archive_rel); return
    free = shutil.disk_usage('/content').free / 1e9
    print(f'\n=== {archive_rel}  (local free {free:.0f} GB)')
    shutil.rmtree('/content/raw', ignore_errors=True)
    os.makedirs('/content/raw/GenVidBench', exist_ok=True)
    path = hf_hub_download(REPO, archive_rel, repo_type='dataset', local_dir='/content/raw/dl')
    pair = archive_rel.split('/')[-2] if '/' in archive_rel else 'Pair1'
    dest = f'/content/raw/GenVidBench/{pair}'
    os.makedirs(dest, exist_ok=True)
    r = subprocess.run(['unar', '-f', '-o', dest, path], capture_output=True, text=True)
    if r.returncode:  # a few corrupt members is normal; the shard builder logs them as missing
        print('unar warning:', r.stdout.strip().splitlines()[-1:])
    n_mp4 = sum(1 for _ in glob.iglob(f'{dest}/**/*.mp4', recursive=True))
    print(f'extracted {n_mp4} videos')
    if n_mp4 == 0:
        raise RuntimeError(f'nothing extracted from {archive_rel}: ' + r.stdout[-500:])
    os.remove(path)  # free disk before sharding
    rc = subprocess.run(['python', 'scripts/build_frame_shards.py', '--manifest', *manifests,
                         '--video-root', '/content/raw/GenVidBench', '--folder', folder,
                         '--out', f'{DRIVE}/frames']).returncode
    shutil.rmtree('/content/raw', ignore_errors=True)
    if rc == 0:
        done.add(archive_rel); json.dump(sorted(done), open(DONE_LOG, 'w'))

PAIR1 = [f for f in files if f.startswith('GenVidBench/Pair1/') and f.endswith('.rar')]
print(PAIR1)
for a in PAIR1:
    process(a, os.path.basename(a)[:-4], MANIFESTS)

## 3. Pair2 → test-B manifest (never trained on)
Needs `Pair2_labels.txt` — check its exact path in the list from step 1.

In [ ]:
lab = [f for f in files if 'Pair2' in f and f.endswith('labels.txt')]
print(lab)
p = hf_hub_download(REPO, lab[0], repo_type='dataset', local_dir='/content/labels')
!python scripts/build_genvidbench_testb.py --labels {p} --out {DRIVE}/manifests/test_b.csv --per-class 2000

## 4. Pair2 archives → shards (only rows in test_b.csv)

In [ ]:
PAIR2 = [f for f in files if f.startswith('GenVidBench/Pair2/') and f.endswith('.rar')]
print(PAIR2)
for a in PAIR2:
    process(a, os.path.basename(a)[:-4], [f'{DRIVE}/manifests/test_b.csv'])

## 5. CLIP ViT-L/14 features (GPU runtime). Resumable.

In [ ]:
!python scripts/extract_clip_features.py --frames {DRIVE}/frames --out {DRIVE}/features

## 6. Summary

In [ ]:
import collections
for root in ['frames', 'features']:
    print(f'--- {root}')
    for g in sorted(os.listdir(f'{DRIVE}/{root}')):
        gd = f'{DRIVE}/{root}/{g}'
        if not os.path.isdir(gd): continue
        if root == 'frames':
            n = sum(len(open(j).read().splitlines()) for j in glob.glob(f'{gd}/*.jsonl'))
        else:
            n = sum(len(json.load(open(j))) for j in glob.glob(f'{gd}/*.ids.json'))
        print(f'{g:20s} {n:7d} videos')
!ls {DRIVE}/frames/failures_*.jsonl 2>/dev/null && wc -l {DRIVE}/frames/failures_*.jsonl
!du -sh {DRIVE}/frames {DRIVE}/features